# Trade-network analysis for Cluster_History

Load the site's normalized `cluster_history_site` membership and only the authorized trade Movement_Vector rows. Direction is retained for degree, paths, and betweenness. Community detection uses an unweighted undirected projection and is exploratory; the generated circular SVG is schematic, not geographic.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from uuid import UUID

from socio_economic.trade_network import (
    analyze_trade_network,
    read_movement_vectors_csv,
    read_participating_sites_csv,
)
from socio_economic.silver_equivalence import sha256_file

cluster_history_id = UUID(os.environ["PLEROMA_CLUSTER_HISTORY_ID"])
analysis_scope = os.environ["PLEROMA_ANALYSIS_SCOPE"]
site_csv = Path(os.environ["PLEROMA_CLUSTER_SITES_CSV"])
movement_vectors_csv = Path(os.environ["PLEROMA_TRADE_VECTORS_CSV"])
trade_category_uri = os.environ["PLEROMA_TRADE_CATEGORY_URI"]
output_dir = Path(os.environ["PLEROMA_OUTPUT_DIR"])
parameter_set_id = UUID(os.environ["PLEROMA_PARAMETER_SET_ID"])
notebook_revision = os.environ["PLEROMA_NOTEBOOK_REVISION"]
environment_reference = os.environ["PLEROMA_ENVIRONMENT_REFERENCE"]
if not output_dir.is_dir():
    raise ValueError("Output directory must already exist")

In [ ]:
site_ids = read_participating_sites_csv(site_csv)
movement_vectors = read_movement_vectors_csv(movement_vectors_csv)
report = analyze_trade_network(
    cluster_history_id=cluster_history_id,
    interaction_type="trade",
    analysis_scope=analysis_scope,
    participating_site_ids=site_ids,
    movement_vectors=movement_vectors,
    trade_category_uri=trade_category_uri,
)
payload_path = output_dir / "trade_network_payload.json"
if payload_path.exists():
    raise FileExistsError(f"Refusing to overwrite {payload_path}")
payload = report.writeback_payload(
    site_membership_sha256=sha256_file(site_csv),
    movement_vectors_sha256=sha256_file(movement_vectors_csv),
    parameter_set_id=parameter_set_id,
    notebook_revision=notebook_revision,
    environment_reference=environment_reference,
    executed_at=datetime.now(timezone.utc),
)
outputs = report.write_outputs(output_dir)
payload_path.write_text(
    json.dumps(payload, allow_nan=False, indent=2) + "\n",
    encoding="utf-8",
)
outputs

Review vector source classes and the original Cluster_History analysis before interpretation. Upload the SVG and its provenance through the authorized writeback workflow, then link it with `asset_role = 'network_graph'`.